# Regression and Comparing Hypothetical Scenarios

This script shows how I arrived at the regression results and the comparisons between the hypothetical scenarios

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
from functools import reduce
import numpy as np
import scipy.stats
from math import sqrt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.graphics.regressionplots as smrp
from libpysal.weights import Queen
import spreg
import patsy
import matplotlib.ticker as mticker

ROOT = Path('../Data')
ROOT.resolve()

results = ROOT/'Spatial Interaction Modelling'
post_processed = ROOT/'Postprocessed'

target_cities = ['Manchester', 'Bristol', 'Leeds']

updated_msoa = gpd.read_file(
    post_processed/'updated_msoa.gpkg'
).to_crs(27700)

### Consolidating all Matrices for all Study Areas for all Scenarios and Sub-Scenarios

In [ ]:
# FORMATTING TRAVEL TIME MATRICES BASED ON RETROSPECTIVE GTFS

full_s0 = []

full_missing = []

for city in target_cities:

    matrix = pd.read_csv(
        results/f'{city}_s0_tt.csv'
    )

    matrix['date'] = matrix['date'].astype(str)

    missing_msoas = matrix.loc[
        matrix['tt'].isna(),
        'msoa21cd'
    ].tolist()

    full_missing.append(missing_msoas)

    cleaned_matrix = matrix[~matrix['msoa21cd'].isin(missing_msoas)]

    actual_perday = cleaned_matrix.groupby(
        ['msoa21cd', 'date']
    ).agg(
        tt = ('tt', 'median')
    ).reset_index().pivot_table(
        index='msoa21cd',
        columns='date',
        values='tt'
    ).reset_index()

    actual_overall = cleaned_matrix.groupby(
        'msoa21cd'
    ).agg(
        s0 = ('tt', 'median'),
        s0_mult = ('tt_mult', 'median')
    ).reset_index()

    baseline = actual_perday.merge(
        actual_overall,
        on='msoa21cd',
        how='inner'
    )

    baseline['TTWA11NM'] = city  # optional but usually useful

    full_s0.append(baseline)

full_baseline = pd.concat(full_s0, ignore_index=True)

In [ ]:
# FORMATTING TRAVEL TIME MATRIX BASED ON SCHEDULED GTFS

full_sched = []

for city in target_cities:

    matrix = pd.read_csv(
        results/f'{city}_scheduled_tt.csv'
    ).groupby(
        'msoa21cd'
    ).agg(
        scheduled = ('tt', 'median')
    ).reset_index()

    cleaned_matrix = matrix[~matrix['msoa21cd'].isin(full_missing)]

    cleaned_matrix['TTWA11NM'] = city

    full_sched.append(cleaned_matrix)

full_scheduled = pd.concat(full_sched, ignore_index=True)

In [ ]:
# FORMATTING TRAVEL TIME MATRICES FOR ALL SCENARIOS AND SUBSCENARIOS

scenarios = [
    's1', 's1_deprived', 's1_inbound',
    's2', 's2_deprived', 's2_inbound',
    's3', 's3_deprived', 's3_inbound'
]

scenario_matrices = {}

for s in scenarios:

    full_scenario = []

    for city in target_cities:

        matrix = pd.read_csv(
            results/f'{city}_{s}_tt.csv'
        ).groupby(
            'msoa21cd'
        ).agg(**{
            s: ('tt', 'median'),
            f'{s}_mult': ('tt_mult', 'median')
        }).reset_index()

        cleaned_matrix = matrix[~matrix['msoa21cd'].isin(full_missing)]

        cleaned_matrix['TTWA11NM'] = city

        full_scenario.append(cleaned_matrix)

    fullscenario_df = pd.concat(full_scenario, ignore_index=True)

    scenario_matrices[s] = fullscenario_df

In [ ]:
# PUTTING ALL TOGETHER, WITH updated_msoa.gpkg

all_matrices = [
    full_baseline,
    full_scheduled,
    *scenario_matrices.values()
]

giant_matrix = reduce(
    lambda left, right: left.merge(
        right,
        on=['msoa21cd', 'TTWA11NM']
    ),
    all_matrices
).rename(
    columns={'msoa21cd': 'MSOA21CD'}
)

giantmat_wmsoa = updated_msoa.merge(
    giant_matrix,
    on=['MSOA21CD', 'TTWA11NM'],
    how='right'
)

In [ ]:
# ADD INFO ABOUT FREQUENCY OF BUSES IN SERVICE AT EACH MSOA

trip_patterns = pd.concat(
    [
        gpd.read_file(post_processed/'Bristol_retro_morning_patterns.gpkg').to_crs(27700),
        gpd.read_file(post_processed/'Leeds_retro_morning_patterns.gpkg').to_crs(27700),
        gpd.read_file(post_processed/'Manchester_retro_morning_patterns.gpkg').to_crs(27700),
    ],
    ignore_index=True
)

pattern_msoa_intersect = gpd.sjoin(
    trip_patterns[['route_id', 'median_trips', 'geometry']],
    giantmat_wmsoa[['MSOA21CD', 'geometry']]
).groupby(
    'MSOA21CD'
).agg(
    median_trips = ('median_trips', 'sum')
).reset_index()

In [ ]:
giant_table = giantmat_wmsoa.merge(
    pattern_msoa_intersect,
    on='MSOA21CD',
    how='left'
)

giant_table['log_tt'] = np.log(giant_table['s0'])
giant_table['log_popcount'] = np.log(giant_table['popcount'])

# Ordered categorical to facilitate regression and plotting
giant_table['imd_group_label'] = pd.Categorical(
    giant_table['imd_group_label'],
    categories=[
        'Most deprived', 
        '20-40 pctile', 
        '40-60 pctile', 
        '60-80 pctile', 
        'Least deprived'
    ],
    ordered=True
)

giant_table['TTWA11NM'] = pd.Categorical(
    giant_table['TTWA11NM'],
    categories=[
        'Manchester',
        'Bristol',
        'Leeds'
    ],
    ordered=True
)

print(giant_table['count'].sum())

### Calibrating for Beta

In [ ]:
def CalcRSquared(observed, estimated):
    """Calculate the r^2 from a series of observed and estimated target values
    inputs:
    Observed: Series of actual observed values
    estimated: Series of predicted values"""
    
    r, p = scipy.stats.pearsonr(observed, estimated)
    R2 = r **2
    
    return R2

def CalcRMSE(observed, estimated):
    """Calculate Root Mean Square Error between a series of observed and estimated values
    inputs:
    Observed: Series of actual observed values
    estimated: Series of predicted values"""
    
    res = (observed - estimated)**2
    RMSE = round(sqrt(res.mean()), 3)
    
    return RMSE

def CalcSRMSE(observed, estimated):
    rmse = sqrt(((observed - estimated) ** 2).mean())
    srmse = round(rmse / observed.mean(), 3)
    return srmse

In [ ]:
calibration = smf.glm(
    formula = 'count ~ log_popcount + log_tt + TTWA11NM',
    data = giant_table,
    family = sm.families.Poisson()
).fit()

beta = abs(calibration.params.iloc[-1])

giant_table['predicted_count'] = calibration.predict(giant_table)
print(f'R-Squared = {CalcRSquared(giant_table["count"], giant_table["predicted_count"])}')
print(f'RMSE = {CalcRMSE(giant_table["count"], giant_table["predicted_count"])}')
print(f'SRMSE = {CalcSRMSE(giant_table["count"], giant_table["predicted_count"])}')
print(f'Sum of All Rows = {giant_table["predicted_count"].sum()}')
print(f'Beta = {beta}')

In [ ]:
print(calibration.summary())

In [ ]:
# Calculating all potential accessiblity values for regressions and analysis later

for_access = [
    '20260603', '20260610', '20260617', '20260624', 
    's0', 's0_mult', 
    'scheduled', 
    's1', 's1_mult', 's1_deprived', 's1_deprived_mult', 's1_inbound', 's1_inbound_mult',
    's2', 's2_mult', 's2_deprived', 's2_deprived_mult', 's2_inbound', 's2_inbound_mult', 
    's3', 's3_mult', 's3_deprived', 's3_deprived_mult', 's3_inbound', 's3_inbound_mult'
]

for col in for_access:
    pot_col = f"{col}_pot"
    giant_table[pot_col] = giant_table['popcount'] * (giant_table[col] ** -beta)

giant_table['tti_rate'] = (giant_table['s0_pot'] - giant_table['scheduled_pot']) / giant_table['s0_pot']

giant_table['ttv_rate'] = (
    giant_table[['20260603_pot', '20260610_pot', '20260617_pot', '20260624_pot']].max(axis=1) - 
    giant_table[['20260603_pot', '20260610_pot', '20260617_pot', '20260624_pot']].min(axis=1)
) / giant_table['s0_pot']

giant_table['log_s0_pot'] = np.log(giant_table['s0_pot'])
giant_table['log_junc_dens'] = np.log(giant_table['junction_dens'])
giant_table['log_stop_dens'] = np.log(giant_table['busstop_dens'])
giant_table['log_median_aadf'] = np.log(giant_table['median_aadf'])
giant_table['log_median_trips'] = np.log(giant_table['median_trips'])


### Assessing Actual Potential Accessibility

In [ ]:
q1 = smf.ols(
    formula = 'log_s0_pot ~ TTWA11NM + minor_rd + b_road + a_road + dual + primary + bus_lane + log_junc_dens + log_stop_dens + log_median_aadf + log_median_trips + median_eucldist_km',
    data = giant_table
).fit()

print(q1.summary())

In [ ]:
q1_v2 = smf.ols(
    formula = 'log_s0_pot ~ minor_rd + b_road + a_road + dual + primary + bus_lane + log_junc_dens + log_stop_dens + log_median_aadf + log_median_trips + median_eucldist_km',
    data = giant_table
).fit()

print(q1_v2.summary())

print(anova_lm(q1_v2, q1))

In [ ]:
q1_v3 = smf.ols(
    formula = 'log_s0_pot ~ minor_rd + b_road + a_road + primary + bus_lane + log_junc_dens + log_stop_dens + log_median_aadf + log_median_trips + median_eucldist_km',
    data = giant_table
).fit()

print(q1_v3.summary())

print(anova_lm(q1_v3, q1_v2))

In [ ]:
w = Queen.from_dataframe(giant_table, use_index=True)

y = giant_table[['log_s0_pot']].values

X_patsy = patsy.dmatrix(
    'minor_rd + b_road + a_road + primary + bus_lane + log_junc_dens + log_stop_dens + log_median_aadf + log_median_trips + median_eucldist_km',
    data=giant_table,
    return_type='dataframe'
).drop(columns='Intercept')

access_spatial = spreg.OLS(
    y, X_patsy.values, w=w,
    spat_diag=True,
    moran=True,
    name_y='log_s0_pot',
    name_x=X_patsy.columns.tolist(),
    name_ds='giant_table'
)
print(access_spatial.summary)

In [ ]:
X = q1_v3.model.exog
vif = pd.DataFrame({
    'variable': q1_v3.model.exog_names,
    'VIF': [variance_inflation_factor(X, i) for i in range(X.shape[1])]
})
print(vif)

In [ ]:
access_error = spreg.GM_Error_Het(
    y, X_patsy.values, w=w, 
    name_y='log_s0_pot', name_x=X_patsy.columns.tolist(),
    name_ds='giant_table'
)
print(access_error.summary)

### Assessing TTI

In [ ]:
q2 = smf.ols(
    formula = 'tti_rate ~ TTWA11NM + minor_rd + b_road + a_road + dual + primary + bus_lane + log_junc_dens + log_stop_dens + log_median_aadf + log_median_trips + median_eucldist_km',
    data = giant_table
).fit()

print(q2.summary())

In [ ]:
q2_v2 = smf.ols(
    formula = 'tti_rate ~ TTWA11NM + minor_rd + b_road + a_road + dual + primary + log_junc_dens + log_stop_dens + log_median_aadf + log_median_trips + median_eucldist_km',
    data = giant_table
).fit()

print(q2_v2.summary())

print(anova_lm(q2_v2, q2))

In [ ]:
q2_v3 = smf.ols(
    formula = 'tti_rate ~ TTWA11NM + minor_rd + a_road + dual + primary + log_junc_dens + log_stop_dens + log_median_aadf + log_median_trips + median_eucldist_km',
    data = giant_table
).fit()

print(q2_v3.summary())

print(anova_lm(q2_v3, q2_v2))

In [ ]:
q2_v4 = smf.ols(
    formula = 'tti_rate ~ TTWA11NM + minor_rd + a_road + primary + log_junc_dens + log_stop_dens + log_median_aadf + log_median_trips + median_eucldist_km',
    data = giant_table
).fit()

print(q2_v4.summary())

print(anova_lm(q2_v4, q2_v3))

In [ ]:
q2_v5 = smf.ols(
    formula = 'tti_rate ~ TTWA11NM + minor_rd + a_road + primary + log_junc_dens + log_stop_dens + log_median_trips + median_eucldist_km',
    data = giant_table
).fit()

print(q2_v5.summary())

print(anova_lm(q2_v5, q2_v4))

In [ ]:
q2_v6 = smf.ols(
    formula = 'tti_rate ~ TTWA11NM + minor_rd + a_road + primary + log_junc_dens + log_stop_dens + log_median_trips',
    data = giant_table
).fit()

print(q2_v6.summary())

print(anova_lm(q2_v6, q2_v5))

In [ ]:
q2_v7 = smf.ols(
    formula = 'tti_rate ~ TTWA11NM + minor_rd + a_road + primary + log_stop_dens + log_median_trips',
    data = giant_table
).fit()

print(q2_v7.summary())

print(anova_lm(q2_v7, q2_v6))

In [ ]:
q2_v8 = smf.ols(
    formula = 'tti_rate ~ TTWA11NM + minor_rd + a_road + primary + log_median_trips',
    data = giant_table
).fit()

print(q2_v8.summary())

print(anova_lm(q2_v8, q2_v7))

In [ ]:
q2_v9 = smf.ols(
    formula = 'tti_rate ~ TTWA11NM + minor_rd + primary + log_median_trips',
    data = giant_table
).fit()

print(q2_v9.summary())

print(anova_lm(q2_v9, q2_v8))

In [ ]:
w = Queen.from_dataframe(giant_table, use_index=True)

y = giant_table[['tti_rate']].values

X_patsy_tti = patsy.dmatrix(
    'TTWA11NM + minor_rd + primary + log_median_trips',
    data=giant_table,
    return_type='dataframe'
).drop(columns='Intercept')

tti_spatial = spreg.OLS(
    y, X_patsy_tti.values, w=w,
    spat_diag=True,
    moran=True,
    name_y='tti_rate',
    name_x=X_patsy_tti.columns.tolist(),
    name_ds='giant_table'
)
print(tti_spatial.summary)

In [ ]:
tti_error = spreg.GM_Error_Het(
    y, X_patsy_tti.values, w=w, 
    name_y='tti_rate', name_x=X_patsy_tti.columns.tolist(),
    name_ds='giant_table'
)
print(tti_error.summary)

### Assessing TTV

In [ ]:
q3 = smf.ols(
    formula = 'ttv_rate ~ TTWA11NM + minor_rd + b_road + a_road + dual + primary + bus_lane + log_junc_dens + log_stop_dens + log_median_aadf + log_median_trips + median_eucldist_km',
    data = giant_table
).fit()

print(q3.summary())

In [ ]:
q3_v2 = smf.ols(
    formula = 'ttv_rate ~ TTWA11NM + minor_rd + b_road + a_road + dual + primary + bus_lane + log_junc_dens + log_stop_dens + log_median_aadf + median_eucldist_km',
    data = giant_table
).fit()

print(q3_v2.summary())

print(anova_lm(q3_v2, q3))

In [ ]:
q3_v3 = smf.ols(
    formula = 'ttv_rate ~ TTWA11NM + minor_rd + b_road + a_road + dual + primary + bus_lane + log_junc_dens + log_median_aadf + median_eucldist_km',
    data = giant_table
).fit()

print(q3_v3.summary())

print(anova_lm(q3_v3, q3_v2))

In [ ]:
q3_v4 = smf.ols(
    formula = 'ttv_rate ~ TTWA11NM + minor_rd + b_road + a_road + primary + bus_lane + log_junc_dens + log_median_aadf + median_eucldist_km',
    data = giant_table
).fit()

print(q3_v4.summary())

print(anova_lm(q3_v4, q3_v3))

In [ ]:
q3_v5 = smf.ols(
    formula = 'ttv_rate ~ TTWA11NM + b_road + a_road + primary + bus_lane + log_junc_dens + log_median_aadf + median_eucldist_km',
    data = giant_table
).fit()

print(q3_v5.summary())

print(anova_lm(q3_v5, q3_v4))

In [ ]:
q3_v6 = smf.ols(
    formula = 'ttv_rate ~ TTWA11NM + b_road + primary + bus_lane + log_junc_dens + log_median_aadf + median_eucldist_km',
    data = giant_table
).fit()

print(q3_v6.summary())

print(anova_lm(q3_v6, q3_v5))

In [ ]:
q3_v7 = smf.ols(
    formula = 'ttv_rate ~ TTWA11NM + b_road + primary + bus_lane + log_median_aadf + median_eucldist_km',
    data = giant_table
).fit()

print(q3_v7.summary())

print(anova_lm(q3_v7, q3_v6))

In [ ]:
w = Queen.from_dataframe(giant_table, use_index=True)

y = giant_table[['ttv_rate']].values

X_patsy_ttv = patsy.dmatrix(
    'TTWA11NM + b_road + primary + bus_lane + log_median_aadf + median_eucldist_km',
    data=giant_table,
    return_type='dataframe'
).drop(columns='Intercept')

ttv_spatial = spreg.OLS(
    y, X_patsy_ttv.values, w=w,
    spat_diag=True,
    moran=True,
    name_y='ttv_rate',
    name_x=X_patsy_ttv.columns.tolist(),
    name_ds='giant_table'
)
print(ttv_spatial.summary)

In [ ]:
X = q3_v7.model.exog
vif = pd.DataFrame({
    'variable': q3_v7.model.exog_names,
    'VIF': [variance_inflation_factor(X, i) for i in range(X.shape[1])]
})
print(vif)

In [ ]:
ttv_error = spreg.GM_Error_Het(
    y, X_patsy_ttv.values, w=w, 
    name_y='ttv_rate', name_x=X_patsy_ttv.columns.tolist(),
    name_ds='giant_table'
)
print(ttv_error.summary)

### Evaluating Hypothetical Scenarios

In [ ]:
for_hypotheticals = [
    's0_mult_pot', 
    's1_pot', 's1_mult_pot', 's1_deprived_pot', 's1_deprived_mult_pot', 's1_inbound_pot', 's1_inbound_mult_pot',
    's2_pot', 's2_mult_pot', 's2_deprived_pot', 's2_deprived_mult_pot', 's2_inbound_pot', 's2_inbound_mult_pot', 
    's3_pot', 's3_mult_pot', 's3_deprived_pot', 's3_deprived_mult_pot', 's3_inbound_pot', 's3_inbound_mult_pot'
]

#### For Bristol

In [ ]:
city_table = giant_table[giant_table['TTWA11NM'] == 'Bristol']

change = []

for col in for_hypotheticals:

    change.append({
        'scenario': col.replace('_pot', ''),
        'potential': city_table[col].sum()
    })

change_table = pd.DataFrame(change)

change_table['pct_change'] = change_table['potential'] / city_table['s0_pot'].sum()

change_table

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

change_table = change_table.sort_values(
    'pct_change', 
    ascending=True
).reset_index()

scenarios = change_table['scenario']
x = range(len(scenarios))

accpot_pct = (change_table['pct_change'] - 1) * 100

ax.bar(x, accpot_pct, color='lightgreen', edgecolor='black')
for level in [10]:
    ax.axhline(level, color='blue', linewidth=1, linestyle=':', alpha=1)
ax.set_xticks(x)
ax.set_xticklabels(scenarios, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('% change from Baseline Conditions')
ax.set_xlabel('Hypothetical Improvement Scenarios/Subscenarios')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

#### For Leeds

In [ ]:
city_table = giant_table[giant_table['TTWA11NM'] == 'Leeds']

change = []

for col in for_hypotheticals:

    change.append({
        'scenario': col.replace('_pot', ''),
        'potential': city_table[col].sum()
    })

change_table = pd.DataFrame(change)

change_table['pct_change'] = change_table['potential'] / city_table['s0_pot'].sum()

change_table

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

change_table = change_table.sort_values(
    'pct_change', 
    ascending=True
).reset_index()

scenarios = change_table['scenario']
x = range(len(scenarios))

accpot_pct = (change_table['pct_change'] - 1) * 100

ax.bar(x, accpot_pct, color='lightgreen', edgecolor='black')
for level in [10, 20]:
    ax.axhline(level, color='blue', linewidth=1, linestyle=':', alpha=1)
ax.set_xticks(x)
ax.set_xticklabels(scenarios, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('% change from Baseline Conditions')
ax.set_xlabel('Hypothetical Improvement Scenarios/Subscenarios')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

#### For Manchester

In [ ]:
city_table = giant_table[giant_table['TTWA11NM'] == 'Manchester']

change = []

for col in for_hypotheticals:

    change.append({
        'scenario': col.replace('_pot', ''),
        'potential': city_table[col].sum()
    })

change_table = pd.DataFrame(change)

change_table['pct_change'] = change_table['potential'] / city_table['s0_pot'].sum()

change_table

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

change_table = change_table.sort_values(
    'pct_change', 
    ascending=True
).reset_index()

scenarios = change_table['scenario']
x = range(len(scenarios))

accpot_pct = (change_table['pct_change'] - 1) * 100

ax.bar(x, accpot_pct, color='lightgreen', edgecolor='black')
for level in [10, 20]:
    ax.axhline(level, color='blue', linewidth=1, linestyle=':', alpha=1)
ax.set_xticks(x)
ax.set_xticklabels(scenarios, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('% change from Baseline Conditions')
ax.set_xlabel('Hypothetical Improvement Scenarios/Subscenarios')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

### Saving Giant Table for Visualisation

In [ ]:
giant_table['time_savings'] = np.select(
    [
        giant_table['TTWA11NM'] == 'Bristol',
        giant_table['TTWA11NM'] == 'Leeds',
        giant_table['TTWA11NM'] == 'Manchester'
    ],
    [
        giant_table['s0'] - giant_table['s2_inbound'],
        giant_table['s0'] - giant_table['s2_deprived'],
        giant_table['s0'] - giant_table['s0_mult']
    ],
    default=np.nan
)

giant_table.to_file(post_processed/'for_visualisation.gpkg', driver='GPKG')